# Brain MRI Classification — Full Pipeline

**One notebook, four stages:**

1. Data prep (dedupe + stratified splits + `common` helpers)
2. ResNet50 (transfer learning)
3. Custom CNN (from scratch)
4. HOG + Linear SVM (classical baseline)

**Re-run safe:** if a model checkpoint already exists under `ROOT/models/`, that stage **loads** it and skips training.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/CSI6209'

import os, sys, time, copy, random, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from skimage.feature import hog
from skimage.color import rgb2gray
import joblib

print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(f'{ROOT}/splits', exist_ok=True)
os.makedirs(f'{ROOT}/models', exist_ok=True)
os.makedirs(f'{ROOT}/results', exist_ok=True)

In [ ]:
# Unzip data only if not already on this runtime
if not os.path.exists('/content/data'):
    !cp /content/drive/MyDrive/CSI6209/brain_mri.zip /content/
    !unzip -q -o /content/brain_mri.zip -d /content/data
    print('Unzipped dataset')
else:
    print('Data already present at /content/data')

!ls /content/data
!ls /content/data/*

## 1. Shared constants & helpers

In [ ]:
SEED = 42
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)
LABEL2ID = {c: i for i, c in enumerate(CLASSES)}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
DATA_DIR = '/content/data'
SPLITS = f'{ROOT}/splits'

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print('Seed set | classes:', CLASSES)

In [ ]:
class BrainMRIDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['label_id'])


train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_loaders(batch_size=32, num_workers=2):
    train_ds = BrainMRIDataset(f'{SPLITS}/train_split.csv', train_tf)
    val_ds   = BrainMRIDataset(f'{SPLITS}/val_split.csv',   eval_tf)
    test_ds  = BrainMRIDataset(f'{SPLITS}/test_split.csv',  eval_tf)
    g = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, generator=g, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        probs = torch.softmax(model(imgs.to(device)), dim=1)
        all_probs.append(probs.cpu().numpy())
        all_preds.append(probs.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
    return (np.concatenate(all_labels),
            np.concatenate(all_preds),
            np.concatenate(all_probs))


def plot_history(history, title, best_epoch, save_path):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax1.plot(epochs, history['train_loss'], 'o-', label='train')
    ax1.plot(epochs, history['val_loss'], 'o-', label='val')
    if best_epoch:
        ax1.axvline(best_epoch, ls=':', c='green', label='best')
    ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, history['train_acc'], 'o-', label='train')
    ax2.plot(epochs, history['val_acc'], 'o-', label='val')
    if best_epoch:
        ax2.axvline(best_epoch, ls=':', c='green', label='best')
    ax2.set_title('Accuracy'); ax2.set_xlabel('epoch'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def report_and_cm(y_true, y_pred, name, save_path):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    print(f'\n=== {name} ===')
    print(f'Test accuracy: {acc:.4f}')
    print(f'Macro F1:      {f1:.4f}\n')
    print(classification_report(y_true, y_pred, target_names=CLASSES))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
        ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name} — test confusion matrix')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return acc, f1

print('Helpers ready')

## 2. Data preparation

Builds train/val/test CSVs once. On re-run, if CSVs already exist, this stage is **skipped**.


In [ ]:
TRAIN_CSV = f'{SPLITS}/train_split.csv'
VAL_CSV   = f'{SPLITS}/val_split.csv'
TEST_CSV  = f'{SPLITS}/test_split.csv'

if os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV) and os.path.exists(TEST_CSV):
    print('Splits already exist — skipping data prep')
    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    test_df  = pd.read_csv(TEST_CSV)
    print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')
else:
    print('Building splits from scratch...')
    rows = []
    for origin in ['Training', 'Testing']:
        for cls in CLASSES:
            folder = os.path.join(DATA_DIR, origin, cls)
            for fname in sorted(os.listdir(folder)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    rows.append({
                        'path': os.path.join(folder, fname),
                        'label': cls,
                        'label_id': LABEL2ID[cls],
                        'origin': origin,
                    })
    df = pd.DataFrame(rows)
    print(f'Total images: {len(df)}')
    print(df.groupby(['origin', 'label']).size().unstack())

    def file_hash(p):
        with open(p, 'rb') as f:
            return hashlib.md5(f.read()).hexdigest()

    df['hash'] = df['path'].apply(file_hash)
    print('Duplicate files within dataset:', df.hash.duplicated().sum())

    dups = df[df.hash.duplicated(keep=False)]
    conflicts = dups.groupby('hash').label.nunique()
    conflict_hashes = conflicts[conflicts > 1].index
    print('Same image with DIFFERENT labels:', len(conflict_hashes))

    before = len(df)
    df_clean = df[~df.hash.isin(conflict_hashes)]
    df_clean = df_clean.drop_duplicates(subset='hash', keep='first').reset_index(drop=True)
    print(f'Removed {before - len(df_clean)} images, {len(df_clean)} remain')

    test_df   = df_clean[df_clean.origin == 'Testing'].reset_index(drop=True)
    train_all = df_clean[df_clean.origin == 'Training'].reset_index(drop=True)
    train_df, val_df = train_test_split(
        train_all, test_size=0.15, stratify=train_all['label_id'], random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)
    print(f'Saved splits → train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

In [ ]:
train_loader, val_loader, test_loader = get_loaders(batch_size=32)
print('Loaders ready | train batches:', len(train_loader),
      '| val:', len(val_loader), '| test:', len(test_loader))

# quick visual check
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col, cls in enumerate(CLASSES):
    samples = train_df[train_df.label == cls].sample(2, random_state=SEED)
    for row in range(2):
        path = samples.iloc[row]['path']
        axes[row, col].imshow(plt.imread(path), cmap='gray')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(cls, fontsize=13)
plt.tight_layout()
plt.savefig(f'{ROOT}/results/sample_grid.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. ResNet50

If `models/resnet50_brain.pth` exists → **load and skip training**.  
Otherwise → train head (3 epochs) then fine-tune layer4 with early stopping.


In [ ]:
RESNET_PATH = f'{ROOT}/models/resnet50_brain.pth'

resnet = models.resnet50(weights='IMAGENET1K_V2')
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(device)

resnet_history = None
resnet_time = None
resnet_best_epoch = None

if os.path.exists(RESNET_PATH):
    ckpt = torch.load(RESNET_PATH, map_location=device, weights_only=False)
    resnet.load_state_dict(ckpt['state_dict'])
    resnet_history = ckpt.get('history')
    resnet_time = ckpt.get('train_time_sec')
    print(f'ResNet50 already trained — loaded from {RESNET_PATH}')
    if resnet_time is not None:
        print(f'  previous train time: {resnet_time/60:.1f} min')
else:
    print('Training ResNet50 from scratch...')
    # --- Stage 1: head only ---
    for p in resnet.parameters():
        p.requires_grad = False
    for p in resnet.fc.parameters():
        p.requires_grad = True

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=1e-3)
    resnet_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()

    for epoch in range(3):
        e0 = time.time()
        tl, ta = train_one_epoch(resnet, train_loader, criterion, optimizer, device)
        vl, va = evaluate(resnet, val_loader, criterion, device)
        for k, v in zip(resnet_history, [tl, vl, ta, va]):
            resnet_history[k].append(v)
        print(f'[head] {epoch+1}/3  train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f} | {time.time()-e0:.0f}s')

    # --- Stage 2: fine-tune layer4 ---
    for p in resnet.layer4.parameters():
        p.requires_grad = True
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, resnet.parameters()), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    best_weights = copy.deepcopy(resnet.state_dict())
    best_epoch, patience, bad_epochs = 0, 3, 0

    for epoch in range(15):
        e0 = time.time()
        tl, ta = train_one_epoch(resnet, train_loader, criterion, optimizer, device)
        vl, va = evaluate(resnet, val_loader, criterion, device)
        scheduler.step(vl)
        for k, v in zip(resnet_history, [tl, vl, ta, va]):
            resnet_history[k].append(v)
        print(f'[fine] {epoch+1}/15  train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f} | {time.time()-e0:.0f}s')

        if vl < best_val_loss:
            best_val_loss, best_epoch, bad_epochs = vl, epoch + 1, 0
            best_weights = copy.deepcopy(resnet.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'Early stopping at fine-epoch {epoch+1}')
                break

    resnet.load_state_dict(best_weights)
    resnet_time = time.time() - t0
    resnet_best_epoch = 3 + best_epoch

    torch.save({
        'state_dict': resnet.state_dict(),
        'classes': CLASSES,
        'train_time_sec': resnet_time,
        'best_epoch': best_epoch,
        'history': resnet_history,
    }, RESNET_PATH)
    print(f'Saved ResNet50 → {RESNET_PATH} ({os.path.getsize(RESNET_PATH)/1e6:.1f} MB)')
    print(f'Train time: {resnet_time/60:.1f} min | best fine-epoch: {best_epoch}')

In [ ]:
if resnet_history is not None:
    be = resnet_best_epoch
    if be is None and resnet_history:
        # approximate if loaded without best_epoch
        be = int(np.argmin(resnet_history['val_loss'])) + 1
    plot_history(resnet_history, 'ResNet50', be,
                 f'{ROOT}/results/resnet50_curves.png')

y_true_r, y_pred_r, _ = get_predictions(resnet, test_loader, device)
acc_r, f1_r = report_and_cm(
    y_true_r, y_pred_r, 'ResNet50',
    f'{ROOT}/results/resnet50_confusion.png')

## 4. Custom CNN

If `models/cnn_brain.pth` exists → **load and skip training**.


In [ ]:
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        # 224 → 112 → 56 → 28 → 14
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


CNN_PATH = f'{ROOT}/models/cnn_brain.pth'
cnn = BrainTumorCNN(num_classes=NUM_CLASSES).to(device)
cnn_history = None
cnn_time = None
cnn_best_epoch = None

if os.path.exists(CNN_PATH):
    ckpt = torch.load(CNN_PATH, map_location=device, weights_only=False)
    cnn.load_state_dict(ckpt['state_dict'])
    cnn_history = ckpt.get('history')
    cnn_time = ckpt.get('train_time_sec')
    cnn_best_epoch = ckpt.get('best_epoch')
    print(f'CNN already trained — loaded from {CNN_PATH}')
    if cnn_time is not None:
        print(f'  previous train time: {cnn_time/60:.1f} min')
else:
    print('Training Custom CNN from scratch...')
    print(f'Total params: {sum(p.numel() for p in cnn.parameters()):,}')
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2)

    cnn_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()
    best_val_loss = float('inf')
    best_weights = copy.deepcopy(cnn.state_dict())
    best_epoch, patience, bad_epochs = 0, 4, 0

    for epoch in range(20):
        e0 = time.time()
        tl, ta = train_one_epoch(cnn, train_loader, criterion, optimizer, device)
        vl, va = evaluate(cnn, val_loader, criterion, device)
        scheduler.step(vl)
        for k, v in zip(cnn_history, [tl, vl, ta, va]):
            cnn_history[k].append(v)
        print(f'epoch {epoch+1}/20  train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f} | {time.time()-e0:.0f}s')

        if vl < best_val_loss:
            best_val_loss, best_epoch, bad_epochs = vl, epoch + 1, 0
            best_weights = copy.deepcopy(cnn.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    cnn.load_state_dict(best_weights)
    cnn_time = time.time() - t0
    cnn_best_epoch = best_epoch

    torch.save({
        'state_dict': cnn.state_dict(),
        'classes': CLASSES,
        'train_time_sec': cnn_time,
        'best_epoch': best_epoch,
        'history': cnn_history,
    }, CNN_PATH)
    print(f'Saved CNN → {CNN_PATH} ({os.path.getsize(CNN_PATH)/1e6:.1f} MB)')
    print(f'Train time: {cnn_time/60:.1f} min | best epoch: {best_epoch}')

In [ ]:
if cnn_history is not None:
    be = cnn_best_epoch
    if be is None and cnn_history:
        be = int(np.argmin(cnn_history['val_loss'])) + 1
    plot_history(cnn_history, 'Custom CNN', be,
                 f'{ROOT}/results/cnn_curves.png')

y_true_c, y_pred_c, _ = get_predictions(cnn, test_loader, device)
acc_c, f1_c = report_and_cm(
    y_true_c, y_pred_c, 'Custom CNN',
    f'{ROOT}/results/cnn_confusion.png')

## 5. HOG + Linear SVM

If `models/hog_svm_brain.joblib` exists → **load and skip training**.


In [ ]:
HOG_PATH = f'{ROOT}/models/hog_svm_brain.joblib'
IMG_SIZE = 128

def extract_hog(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    gray = rgb2gray(np.array(img))
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True,
    )

hog_time = None

if os.path.exists(HOG_PATH):
    blob = joblib.load(HOG_PATH)
    hog_pipe = blob['pipeline']
    hog_time = blob.get('train_time_sec')
    print(f'HOG+SVM already trained — loaded from {HOG_PATH}')
    if hog_time is not None:
        print(f'  previous fit time: {hog_time:.1f}s')
else:
    print('Extracting HOG features (a few minutes)...')
    train_all = pd.concat([train_df, val_df], ignore_index=True)
    t0 = time.time()
    X_train = np.stack([extract_hog(p) for p in train_all['path']])
    y_train = train_all['label_id'].values
    print(f'Feature dim: {X_train.shape[1]} | extract time: {(time.time()-t0)/60:.1f} min')

    hog_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', LinearSVC(C=1.0, max_iter=5000, dual='auto', random_state=SEED)),
    ])
    t1 = time.time()
    hog_pipe.fit(X_train, y_train)
    hog_time = time.time() - t1
    print(f'SVM fit time: {hog_time:.1f}s')

    joblib.dump({
        'pipeline': hog_pipe,
        'classes': CLASSES,
        'img_size': IMG_SIZE,
        'train_time_sec': hog_time,
    }, HOG_PATH)
    print(f'Saved HOG+SVM → {HOG_PATH}')

In [ ]:
print('Extracting HOG on test set...')
X_test = np.stack([extract_hog(p) for p in test_df['path']])
y_test = test_df['label_id'].values
y_pred_h = hog_pipe.predict(X_test)

acc_h, f1_h = report_and_cm(
    y_test, y_pred_h, 'HOG + SVM',
    f'{ROOT}/results/hog_svm_confusion.png')

## 6. Final comparison

In [ ]:
summary = pd.DataFrame([
    {
        'model': 'ResNet50',
        'test_acc': acc_r,
        'macro_f1': f1_r,
        'train_min': round(resnet_time / 60, 1) if resnet_time else None,
    },
    {
        'model': 'Custom CNN',
        'test_acc': acc_c,
        'macro_f1': f1_c,
        'train_min': round(cnn_time / 60, 1) if cnn_time else None,
    },
    {
        'model': 'HOG + SVM',
        'test_acc': acc_h,
        'macro_f1': f1_h,
        'train_min': round(hog_time / 60, 2) if hog_time else None,
    },
])
print(summary.to_string(index=False))
summary.to_csv(f'{ROOT}/results/full_comparison.csv', index=False)
print(f'\nSaved → {ROOT}/results/full_comparison.csv')
print('Pipeline complete.')